<img src="https://raw.githubusercontent.com/OmarLi-360/powerbi-tableau-analista/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Clave 4 · Medidas DAX y el contexto de filtro
**Analista de datos · Power BI + Tableau** — Cifras de control para tus medidas · ⏱ 30 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/powerbi-tableau-analista/blob/main/notebooks/04_medidas_dax.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/powerbi-tableau-analista/#clave-04)

<sub>Si al abrir un enlace Google muestra «Redirect Notice», haz clic en el enlace azul de esa página: es un aviso de seguridad de Google, no un error.</sub>

**Al terminar podrás:**
1. Obtener la tabla de control de las seis medidas para cualquier año y sede.
2. Ver cómo una misma medida cambia según el contexto de filtro.
3. Entender por qué un porcentaje no se suma en el total.

---
### Cómo usar este cuaderno (no necesitas programar)
1. **Guarda tu copia:** *Archivo › Guardar una copia en Drive*. Así tus cambios no se pierden. Si no lo haces, no pasa nada grave: el original está protegido y nadie puede dañarlo desde aquí, pero lo que cambies se borra al cerrar.
2. **Ejecuta todo:** *Entorno de ejecución › Ejecutar todas* (Ctrl+F9). Si Colab avisa que el cuaderno no es de Google, elige *Ejecutar de todos modos*.
3. **Juega con los controles:** las celdas con listas desplegables a la derecha se pueden cambiar. Elige una opción y pulsa el botón ▶ de esa celda para redibujar.

El código está oculto a propósito: aquí importa entender el dato y el gráfico. Si tienes curiosidad, haz doble clic en una celda para verlo.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/powerbi-tableau-analista"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

In [ ]:
# @title Definir las seis medidas (las mismas de DAX)
def medidas(v, m):
    """v: líneas de venta con el filtro de la celda; m: metas con el mismo filtro."""
    ok = v[v.estado != "Anulada"]                      # CALCULATE(..., Ventas[estado] <> "Anulada")
    venta, fact, meta = ok.subtotal.sum(), ok.id_factura.nunique(), m.meta_subtotal.sum()
    return {"Venta neta": venta, "Meta": meta,
            "Cumplimiento": venta / meta if meta else np.nan,          # DIVIDE([Venta neta], [Meta])
            "Facturas": fact,                                            # DISTINCTCOUNT(id_factura)
            "Ticket promedio": venta / fact if fact else np.nan,         # DIVIDE([Venta neta], [Facturas])
            "Margen %": ok.margen_bruto.sum() / venta if venta else np.nan}
ventas_ene = pd.read_csv(ruta("nuevo_mes/ventas_2026_01.csv"), parse_dates=["fecha"])
metas_ene = pd.read_csv(ruta("nuevo_mes/metas_2026_01.csv"))
V = (pd.concat([ventas, ventas_ene]).merge(sedes[["id_sede", "ciudad"]], on="id_sede")
     .merge(clientes[["id_cliente", "segmento"]], on="id_cliente").assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
M = pd.concat([metas, metas_ene]).merge(sedes[["id_sede", "ciudad"]], on="id_sede")
def formato(nombre, x):
    if pd.isna(x): return "(vacío)"
    return porc(x) if nombre in ("Cumplimiento", "Margen %") else miles(x)
display(Markdown("Medidas listas: Venta neta, Meta, Cumplimiento, Facturas, Ticket promedio y Margen %."))

## 1 · Tu tabla de control
Elige el año y la sede **igual que en las segmentaciones de tu Power BI**. Las seis cifras deben coincidir con tus tarjetas. Si una no coincide, el problema está en la medida o en el modelo, no en el visual.

In [ ]:
# @title Tabla de control
anio = "2025"  # @param ["2024", "2025", "Enero 2026"]
sede = "Todas"  # @param ["Todas", "Bogotá", "Medellín", "Cali", "Barranquilla", "Bucaramanga", "Pereira"]
fv = V.anio == (2026 if anio == "Enero 2026" else int(anio))
fm = M.anio == (2026 if anio == "Enero 2026" else int(anio))
if sede != "Todas":
    fv, fm = fv & (V.ciudad == sede), fm & (M.ciudad == sede)
control = medidas(V[fv], M[fm])
display(pd.DataFrame({"Medida": control.keys(), "Debe dar": [formato(k, x) for k, x in control.items()]}).style.hide(axis="index"))
display(Markdown(f"Filtro equivalente en Power BI: **Calendario[Año] = {anio.replace('Enero ', '')}**" + (f" y **Sedes[ciudad] = {sede}**" if sede != "Todas" else "")
                 + (" y **Calendario[Mes] = 1**" if anio == "Enero 2026" else "")))

## 2 · El contexto de filtro

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cómo cambia la medida en cada fila de la matriz? |
> | **Qué se grafica** | Barras por sede con la medida elegida · Línea punteada: el valor del total |
> | **Por qué esta forma** | Cada barra es una celda de la matriz con su propio filtro de sede; la línea es la misma medida sin ese filtro. |
> | **Qué se descartó** | Sumar las barras para obtener el total: solo funciona con medidas aditivas como la venta. |
> | **Cómo se lee** | Para Venta neta y Facturas las barras suman el total. Para Ticket y Cumplimiento, el total es otro cálculo: no es la suma ni el promedio de las barras. |
> | **En Excel 365** | Matriz con Sedes[ciudad] en filas y la medida en valores; mira la fila Total. |

In [ ]:
# @title Gráfico 1 · Una medida, seis contextos
medida = "Ticket promedio"  # @param ["Venta neta", "Facturas", "Ticket promedio", "Cumplimiento"]
anio_g = 2025
filas_s = {c: medidas(V[(V.anio == anio_g) & (V.ciudad == c)], M[(M.anio == anio_g) & (M.ciudad == c)])[medida] for c in ORDEN_CIUDADES}
total = medidas(V[V.anio == anio_g], M[M.anio == anio_g])[medida]
s = pd.Series(filas_s)
pct = medida == "Cumplimiento"
fig = go.Figure(go.Bar(x=s.index, y=s.values, marker_color=AZUL, text=[formato(medida, v) for v in s.values], textposition="outside"))
fig.add_hline(y=total, line_dash="dash", line_color=MAGENTA, annotation_text=f"Total: {formato(medida, total)}", annotation_position="top left")
suma = s.sum()
nota = (f"el total ({formato(medida, total)}) es la suma de las filas" if abs(suma - total) < 1
        else f"el total ({formato(medida, total)}) no es la suma ({formato(medida, suma)}) ni el promedio ({formato(medida, s.mean())}) de las filas")
fig.update_layout(title=f"{medida} 2025 por sede: {nota}", yaxis_tickformat=".0%" if pct else None, height=380,
                  yaxis_range=[0, max(s.max(), total) * 1.2])
fig.show()

## 3 · El porcentaje que se suma

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Qué pasa si el cumplimiento es una columna calculada? |
> | **Qué se grafica** | Barras: cumplimiento por sede y dos totales: el de la columna sumada y el de la medida |
> | **Por qué esta forma** | Pone lado a lado el total equivocado y el correcto, con las mismas filas. |
> | **Qué se descartó** | Mostrar solo el total correcto: el error no se ve. |
> | **Cómo se lee** | La barra magenta suma porcentajes: no significa nada. La azul divide la venta total entre la meta total. |
> | **En Excel 365** | Compara una columna calculada con la medida [Cumplimiento] en la misma matriz. |

In [ ]:
# @title Gráfico 2 · Columna sumada frente a medida
cum = pd.Series({c: medidas(V[(V.anio == 2025) & (V.ciudad == c)], M[(M.anio == 2025) & (M.ciudad == c)])["Cumplimiento"] for c in ORDEN_CIUDADES})
total_medida = medidas(V[V.anio == 2025], M[M.anio == 2025])["Cumplimiento"]
etiquetas = list(cum.index) + ["Total: columna sumada", "Total: medida"]
valores = list(cum.values) + [cum.sum(), total_medida]
fig = go.Figure(go.Bar(x=etiquetas, y=valores, marker_color=[GRIS] * len(cum) + [MAGENTA, AZUL],
                       text=[porc(v) for v in valores], textposition="outside"))
fig.update_layout(title=f"Sumar porcentajes da {porc(cum.sum())}; la medida da {porc(total_medida)}",
                  yaxis_tickformat=".0%", height=380, yaxis_range=[0, cum.sum() * 1.15])
fig.show()

## 4 · Retos de interpretación

In [ ]:
# @title Reto 1 · En 2025, ¿cuál es la venta neta de Cali?
respuesta = "Elige una opción"  # @param ["Elige una opción", "2.348.175.015", "2.391.555.945", "2.040.095.660", "4.438.208.370"]
revisar(respuesta, miles(medidas(V[(V.anio == 2025) & (V.ciudad == "Cali")], M[(M.anio == 2025) & (M.ciudad == "Cali")])["Venta neta"]), """Usa la tabla de control con año 2025 y sede Cali. La segunda opción incluye anuladas.""")

In [ ]:
# @title Reto 2 · El ticket promedio total de 2025, ¿es el promedio de los tickets de las sedes?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Sí, siempre", "No: es la venta total dividida entre las facturas totales", "Solo si hay seis sedes", "Depende del visual"]
revisar(respuesta, "No: es la venta total dividida entre las facturas totales", """La medida se evalúa en el contexto del total, sin filtro de sede.""")

In [ ]:
# @title Reto 3 · ¿Qué medida da 100 % en cada fila si se escribe sin ALL?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Venta neta", "Participación", "Facturas", "Meta"]
revisar(respuesta, "Participación", """Sin ALL, el numerador y el denominador tienen el mismo filtro de sede.""")

## Apóyate en IA
Para escribir una medida nueva y verificarla:

```text
En Power BI tengo Ventas (subtotal, estado, id_factura, margen_bruto) relacionada con Sedes y Calendario.
Escribe en DAX la medida [NOMBRE] sin facturas anuladas y dime dos valores de prueba que pueda comparar con una tabla de control.
```

> **Verifica siempre.** Compara la respuesta de la IA con un dato que ya conozcas. No pegues datos personales reales (Ley 1581 de 2012): estos datos son ficticios, los de tu empresa no.

## Resumen
* Una medida se calcula para cada celda con su propio contexto de filtro.
* El total no es la suma de las filas: es la misma medida con menos filtros.
* Porcentajes, promedios y razones van en medidas, nunca en columnas que se suman.

**Siguiente:** [Clave 6 · Informe que responde preguntas](https://omarli-360.github.io/powerbi-tableau-analista/#clave-06) · [Cuaderno 06](https://colab.research.google.com/github/OmarLi-360/powerbi-tableau-analista/blob/main/notebooks/06_informe_que_responde.ipynb)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>